In [ ]:
import os
import random
import shutil
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import glob
from sklearn.model_selection import train_test_split
import torch
import warnings
warnings.filterwarnings('ignore')

from ultralytics import YOLO

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    memory_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU Memory: {memory_gb:.1f} GB")

Ultralytics YOLO already installed
PyTorch version: 2.6.0+cu124
CUDA available: True
Number of GPUs available: 8
GPU 0: NVIDIA GeForce RTX 3090
  Memory: 23.7 GB
GPU 1: NVIDIA GeForce RTX 3090
  Memory: 23.7 GB
GPU 2: NVIDIA GeForce RTX 3090
  Memory: 23.7 GB
GPU 3: NVIDIA GeForce RTX 3090
  Memory: 23.7 GB
GPU 4: NVIDIA GeForce RTX 3090
  Memory: 23.7 GB
GPU 5: NVIDIA GeForce RTX 3090
  Memory: 23.7 GB
GPU 6: NVIDIA GeForce RTX 3090
  Memory: 23.7 GB
GPU 7: NVIDIA GeForce RTX 3090
  Memory: 23.7 GB

Multi-GPU setup detected! Will use 8 GPUs for training.
Available device IDs: [0, 1, 2, 3, 4, 5, 6, 7]
Ultralytics version: 8.3.168


In [ ]:
def load_dataset(data_dir):
    """Load dataset from all country directories"""
    dataset = []
    country_dirs = [d for d in os.listdir(data_dir) if d.startswith('country_')]
    
    for country_dir in country_dirs:
        country_path = os.path.join(data_dir, country_dir)
        images_path = os.path.join(country_path, 'images')
        labels_path = os.path.join(country_path, 'labels')

        image_files = glob.glob(os.path.join(images_path, '*.jpg'))

        for image_file in image_files:
            base_name = os.path.splitext(os.path.basename(image_file))[0]
            label_file = os.path.join(labels_path, f"{base_name}.txt")

            if os.path.exists(label_file):
                bboxes = []
                with open(label_file, 'r') as f:
                    for line in f:
                        line = line.strip()
                        if line:
                            parts = line.split()
                            if len(parts) == 5:
                                class_id = int(parts[0])
                                x_center = float(parts[1])
                                y_center = float(parts[2])
                                width = float(parts[3])
                                height = float(parts[4])
                                bboxes.append({
                                    'class_id': class_id,
                                    'x_center': x_center,
                                    'y_center': y_center,
                                    'width': width,
                                    'height': height,
                                })

                dataset.append({
                    'image_path': image_file,
                    'label_path': label_file,
                    'bboxes': bboxes,
                    'country': country_dir
                })

    return dataset

# Load dataset
data_dir = "data"
dataset = load_dataset(data_dir)

# Define class names
class_names = {
    0: 'Pothole',
    1: 'Alligator Crack',
    2: 'Transverse Crack',
    3: 'Longitudinal Crack',
}

# Basic statistics
samples_with_bboxes = [sample for sample in dataset if len(sample['bboxes']) > 0]
class_counts = {}
for sample in samples_with_bboxes:
    for bbox in sample['bboxes']:
        class_id = bbox['class_id']
        class_counts[class_id] = class_counts.get(class_id, 0) + 1

print(f"Total samples: {len(dataset)}")
print(f"Samples with annotations: {len(samples_with_bboxes)}")
print("Class distribution:")
for class_id, count in sorted(class_counts.items()):
    class_name = class_names.get(class_id, f"Class {class_id}")
    print(f"  {class_name}: {count}")

Found 3 country directories: ['country_2', 'country_3', 'country_1']
Found 2082 images in country_2
Found 1981 images in country_3


Found 1976 images in country_1

Total samples loaded: 6039
=== Dataset Statistics ===

Samples by country:
  country_1: 1976 samples
  country_2: 2082 samples
  country_3: 1981 samples

Total bounding boxes: 16238
Average bounding boxes per image: 2.69

Class distribution:
  Pothole: 3425 (21.1%)
  Alligator Crack: 3582 (22.1%)
  Transverse Crack: 4280 (26.4%)
  Longitudinal Crack: 4951 (30.5%)

Samples with bounding boxes: 6039
Samples without bounding boxes: 0


In [ ]:
def prepare_yolo_dataset(dataset, output_dir="yolo_dataset", train_ratio=0.8, val_ratio=0.1):
    """Prepare dataset in YOLO format with train/val/test splits"""
    # Create output directory structure
    for split in ['train', 'val', 'test']:
        os.makedirs(os.path.join(output_dir, split, 'images'), exist_ok=True)
        os.makedirs(os.path.join(output_dir, split, 'labels'), exist_ok=True)
    
    # Split dataset
    train_samples, temp_samples = train_test_split(
        samples_with_bboxes, test_size=1-train_ratio, random_state=42
    )
    val_samples, test_samples = train_test_split(
        temp_samples, test_size=val_ratio/(val_ratio + (1-train_ratio-val_ratio)), random_state=42
    )
    
    splits = {
        'train': train_samples,
        'val': val_samples,
        'test': test_samples
    }
    
    print(f"Train: {len(train_samples)}, Val: {len(val_samples)}, Test: {len(test_samples)}")
    
    # Copy files to YOLO format
    for split_name, samples in splits.items():
        for sample in samples:
            # Copy image and label
            image_name = os.path.basename(sample['image_path'])
            label_name = os.path.basename(sample['label_path'])
            
            new_image_path = os.path.join(output_dir, split_name, 'images', image_name)
            new_label_path = os.path.join(output_dir, split_name, 'labels', label_name)
            
            shutil.copy2(sample['image_path'], new_image_path)
            shutil.copy2(sample['label_path'], new_label_path)
    
    return splits, output_dir

# Prepare YOLO dataset
splits, yolo_dataset_dir = prepare_yolo_dataset(dataset)

# Create YAML config file
yaml_config = {
    'path': os.path.abspath(yolo_dataset_dir),
    'train': 'train/images',
    'val': 'val/images',
    'test': 'test/images',
    'nc': len(class_names),
    'names': list(class_names.values())
}

yaml_path = os.path.join(yolo_dataset_dir, 'data.yaml')
with open(yaml_path, 'w') as f:
    yaml.dump(yaml_config, f)

print(f"Dataset prepared: {yolo_dataset_dir}")
print(f"Config file: {yaml_path}")

=== Preparing YOLO Dataset ===
Dataset splits:
  Train: 4831 samples
  Val: 603 samples
  Test: 605 samples

Preparing train set...
  Processed 100/4831 samples
  Processed 200/4831 samples
  Processed 300/4831 samples
  Processed 400/4831 samples
  Processed 500/4831 samples


  Processed 600/4831 samples
  Processed 700/4831 samples
  Processed 800/4831 samples
  Processed 900/4831 samples
  Processed 1000/4831 samples
  Processed 1100/4831 samples
  Processed 1200/4831 samples
  Processed 1300/4831 samples
  Processed 1400/4831 samples
  Processed 900/4831 samples
  Processed 1000/4831 samples
  Processed 1100/4831 samples
  Processed 1200/4831 samples
  Processed 1300/4831 samples
  Processed 1400/4831 samples
  Processed 1500/4831 samples
  Processed 1600/4831 samples
  Processed 1700/4831 samples
  Processed 1500/4831 samples
  Processed 1600/4831 samples
  Processed 1700/4831 samples
  Processed 1800/4831 samples
  Processed 1900/4831 samples
  Processed 2000/4831 samples
  Processed 2100/4831 samples
  Processed 2200/4831 samples
  Processed 2300/4831 samples
  Processed 1800/4831 samples
  Processed 1900/4831 samples
  Processed 2000/4831 samples
  Processed 2100/4831 samples
  Processed 2200/4831 samples
  Processed 2300/4831 samples
  Processed 240

In [ ]:
# Initialize YOLO11 model
model = YOLO('yolo11n.pt')

# Configure training parameters
if torch.cuda.is_available():
    device = 'cuda:0'
    batch_size = 32
    workers = 8
else:
    device = 'cpu'
    batch_size = 8
    workers = 4

# Training configuration
training_config = {
    'data': yaml_path,
    'epochs': 100,
    'batch': batch_size,
    'imgsz': 640,
    'device': device,
    'workers': workers,
    'project': 'road_damage_detection',
    'name': 'yolo11_training',
    'save_period': 10,
    'patience': 15,
    'optimizer': 'AdamW',
    'lr0': 0.01,
    'lrf': 0.01,
    'momentum': 0.937,
    'weight_decay': 0.0005,
    'warmup_epochs': 3,
    'augment': True,
    'mosaic': 1.0,
    'mixup': 0.0,
    'fliplr': 0.5,
    'hsv_h': 0.015,
    'hsv_s': 0.7,
    'hsv_v': 0.4,
    'resume': False,
    'exist_ok': True,
    'verbose': True,
    'amp': True,
}

print(f"Model: YOLO11n")
print(f"Device: {device}")
print(f"Batch size: {batch_size}")
print("Ready for training")

=== Initializing YOLO11 Model ===
Loaded YOLO11 model: yolo11n.pt
Single GPU Configuration (avoiding MKL issues):
  Using GPU: 0
  Batch size: 32
  Workers: 8

YOLO11 Single GPU Training Configuration:
  data: yolo_dataset/data.yaml
  epochs: 100
  batch: 32
  imgsz: 640
  device: 0
  workers: 8
  project: road_damage_detection
  name: yolo11_single_gpu
  save_period: 10
  patience: 15
  optimizer: AdamW
  lr0: 0.01
  lrf: 0.01
  momentum: 0.937
  weight_decay: 0.0005
  warmup_epochs: 3
  warmup_momentum: 0.8
  warmup_bias_lr: 0.1
  box: 7.5
  cls: 0.5
  dfl: 1.5
  augment: True
  mosaic: 1.0
  mixup: 0.0
  copy_paste: 0.0
  degrees: 0.0
  translate: 0.1
  scale: 0.5
  shear: 0.0
  perspective: 0.0
  flipud: 0.0
  fliplr: 0.5
  hsv_h: 0.015
  hsv_s: 0.7
  hsv_v: 0.4
  resume: False
  exist_ok: True
  verbose: True
  amp: True
  fraction: 1.0
  close_mosaic: 10
  cos_lr: False

GPU 0 memory cleared

=== YOLO11 Model Ready for Single GPU Training ===
Benefits of single GPU approach:
- No

In [ ]:
# Start YOLO11 training
print("Starting YOLO11 training...")

try:
    results = model.train(**training_config)
    print(f"Training completed successfully!")
    print(f"Results saved at: {results.save_dir}")
    
    # Clean up GPU memory
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        
except Exception as e:
    print(f"Training failed: {e}")
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    raise e

=== Starting YOLO11 Single GPU Training ===
Training Setup:
  Model: YOLO11 (yolo11n.pt)
  Device: GPU 0
  Batch size: 32
  Workers: 8
  Learning rate: 0.0100
  Image size: 640
  AMP enabled: True

GPU Memory Status Before Training:
  GPU 0: 0.0GB / 23.7GB allocated
Starting single GPU training (avoiding distributed training issues)...
This approach is more stable and avoids MKL errors.

🚀 Training YOLO11 with single GPU configuration...
Ultralytics 8.3.168 🚀 Python-3.13.5 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 3090, 24253MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=True, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=yolo_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchs

  8                  -1  1    346112  ultralytics.nn.modules.block.C3k2            [256, 256, 1, True]           
  9                  -1  1    164608  ultralytics.nn.modules.block.SPPF            [256, 256, 5]                 
 10                  -1  1    249728  ultralytics.nn.modules.block.C2PSA           [256, 256, 1]                 
 11                  -1  1         0  torch.nn.modules.upsampling.Upsample         [None, 2, 'nearest']          
 12             [-1, 6]  1         0  ultralytics.nn.modules.conv.Concat           [1]                           
 13                  -1  1    111296  ultralytics.nn.modules.block.C3k2            [384, 128, 1, False]          
 14                  -1  1         0  torch.nn.modules.upsampling.Upsample         [None, 2, 'nearest']          
 15             [-1, 4]  1         0  ultralytics.nn.modules.conv.Concat           [1]                           
 16                  -1  1     32096  ultralytics.nn.modules.block.C3k2            [256,

train: Scanning /home/kwdahun/2025IMSC-Hackathon-CRACKPINK/yolo_dataset/train/labels.cache... 4831 images, 0 backgrounds, 0 corrupt: 100%|██████████| 4831/4831 [00:00<?, ?it/s]
train: Scanning /home/kwdahun/2025IMSC-Hackathon-CRACKPINK/yolo_dataset/train/labels.cache... 4831 images, 0 backgrounds, 0 corrupt: 100%|██████████| 4831/4831 [00:00<?, ?it/s]


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1362.2±731.2 MB/s, size: 66.7 KB)


val: Scanning /home/kwdahun/2025IMSC-Hackathon-CRACKPINK/yolo_dataset/val/labels.cache... 603 images, 0 backgrounds, 0 corrupt: 100%|██████████| 603/603 [00:00<?, ?it/s]



...

      2/100      5.13G       2.46       3.11      2.212        135        640: 100%|██████████| 151/151 [00:19<00:00,  7.79it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95):   0%|          | 0/10 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 10/10 [00:01<00:00,  5.33it/s]

                   all        603       1557     0.0514      0.078     0.0165    0.00539



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


      3/100      5.13G       2.36      2.977      2.154        135        640:  44%|████▎     | 66/151 [00:08<00:10,  7.95it/s]
Exception in thread Thread-6 (_pin_memory_loop):
Traceback (most recent call last):
  File "/home/kwdahun/anaconda3/lib/python3.13/threading.py", line 1043, in _bootstrap_inner
    self.run()
    ~~~~~~~~^^
  File "/home/kwdahun/anaconda3/lib/python3.13/site-packages/ipykernel/ipkernel.py", line 766, in run_closure
    _threading_Thread_run(self)
    ~~~~~~~~~~~~~~~~~~~~~^^^^^^
  File "/home/kwdahun/anaconda3/lib/python3.13/threading.py", line 994, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/kwdahun/anaconda3/lib/python3.13/site-packages/torch/utils/data/_utils/pin_memory.py", line 59, in _pin_memory_loop
    do_one_step()
    ~~~~~~~~~~~^^
  File "/home/kwdahun/anaconda3/lib/python3.13/site-packages/torch/utils/data/_utils/pin_memory.py", line 35, in do_one_step
    r = in_queue.get(timeout=

KeyboardInterrupt: 

In [ ]:
# Load best model and evaluate
best_model_path = os.path.join(results.save_dir, 'weights', 'best.pt')
best_model = YOLO(best_model_path)

# Validate on validation set
validation_results = best_model.val(data=yaml_path, split='val')

# Test on test set
test_results = best_model.val(data=yaml_path, split='test')

# Display metrics
print("Validation Metrics:")
print(f"  mAP@0.5: {validation_results.box.map50:.4f}")
print(f"  mAP@0.5:0.95: {validation_results.box.map:.4f}")
print(f"  Precision: {validation_results.box.mp:.4f}")
print(f"  Recall: {validation_results.box.mr:.4f}")

print("\nPer-class AP@0.5:")
for i, class_name in enumerate(class_names.values()):
    if i < len(validation_results.box.ap50):
        ap50 = validation_results.box.ap50[i]
        print(f"  {class_name}: {ap50:.4f}")

In [ ]:
def visualize_predictions(model, test_images, confidence_threshold=0.5, max_images=6):
    """Visualize model predictions on test images"""
    test_image_paths = random.sample(test_images, min(max_images, len(test_images)))
    
    n_cols = 3
    n_rows = (len(test_image_paths) + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5*n_rows))
    axes = axes.flatten() if len(test_image_paths) > 1 else [axes]
    
    colors = ['red', 'blue', 'green', 'yellow']
    
    for i, image_path in enumerate(test_image_paths):
        results = model.predict(image_path, conf=confidence_threshold)
        
        img = Image.open(image_path)
        axes[i].imshow(img)
        
        # Draw predictions
        if len(results) > 0 and results[0].boxes is not None:
            boxes = results[0].boxes
            for box in boxes:
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                conf = box.conf[0].cpu().numpy()
                cls = int(box.cls[0].cpu().numpy())
                
                rect = plt.Rectangle((x1, y1), x2-x1, y2-y1, 
                                   fill=False, color=colors[cls % len(colors)], 
                                   linewidth=2)
                axes[i].add_patch(rect)
                
                class_name = list(class_names.values())[cls]
                axes[i].text(x1, y1-10, f'{class_name}: {conf:.2f}',
                           bbox=dict(boxstyle="round,pad=0.3", 
                                   facecolor=colors[cls % len(colors)], 
                                   alpha=0.8),
                           fontsize=8, color='white', fontweight='bold')
        
        axes[i].set_title(f'{os.path.basename(image_path)}')
        axes[i].axis('off')
    
    # Hide unused subplots
    for i in range(len(test_image_paths), len(axes)):
        axes[i].axis('off')
    
    plt.tight_layout()
    plt.show()

# Visualize predictions on test images
test_images = [sample['image_path'] for sample in splits['test']]
visualize_predictions(best_model, test_images, confidence_threshold=0.3)

In [ ]:
# Visualize training results
results_csv = os.path.join(results.save_dir, 'results.csv')

if os.path.exists(results_csv):
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()
    
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    
    # Plot losses
    axes[0, 0].plot(df['epoch'], df['train/box_loss'], label='Train', color='blue')
    axes[0, 0].plot(df['epoch'], df['val/box_loss'], label='Val', color='red')
    axes[0, 0].set_title('Box Loss')
    axes[0, 0].legend()
    axes[0, 0].grid(True)
    
    axes[0, 1].plot(df['epoch'], df['train/cls_loss'], label='Train', color='blue')
    axes[0, 1].plot(df['epoch'], df['val/cls_loss'], label='Val', color='red')
    axes[0, 1].set_title('Classification Loss')
    axes[0, 1].legend()
    axes[0, 1].grid(True)
    
    # Plot metrics
    axes[1, 0].plot(df['epoch'], df['metrics/mAP50(B)'], label='mAP@0.5', color='green')
    axes[1, 0].plot(df['epoch'], df['metrics/mAP50-95(B)'], label='mAP@0.5:0.95', color='orange')
    axes[1, 0].set_title('mAP')
    axes[1, 0].legend()
    axes[1, 0].grid(True)
    
    axes[1, 1].plot(df['epoch'], df['metrics/precision(B)'], label='Precision', color='purple')
    axes[1, 1].plot(df['epoch'], df['metrics/recall(B)'], label='Recall', color='brown')
    axes[1, 1].set_title('Precision & Recall')
    axes[1, 1].legend()
    axes[1, 1].grid(True)
    
    plt.tight_layout()
    plt.show()
    
    # Final metrics
    final_metrics = df.iloc[-1]
    print("Final Metrics:")
    print(f"  mAP@0.5: {final_metrics['metrics/mAP50(B)']:.4f}")
    print(f"  mAP@0.5:0.95: {final_metrics['metrics/mAP50-95(B)']:.4f}")
    print(f"  Precision: {final_metrics['metrics/precision(B)']:.4f}")
    print(f"  Recall: {final_metrics['metrics/recall(B)']:.4f}")
else:
    print("Training results not found")

In [ ]:
# Export model
print("Exporting model...")
try:
    best_model.export(format='onnx')
    print("Model exported to ONNX format")
except Exception as e:
    print(f"Export failed: {e}")

# Save model info
model_info = {
    'model': 'yolo11n.pt',
    'classes': class_names,
    'best_model_path': best_model_path,
    'dataset_path': yolo_dataset_dir,
    'metrics': {
        'mAP50': float(validation_results.box.map50),
        'mAP50-95': float(validation_results.box.map),
        'precision': float(validation_results.box.mp),
        'recall': float(validation_results.box.mr)
    }
}

info_path = os.path.join(results.save_dir, 'model_info.yaml')
with open(info_path, 'w') as f:
    yaml.dump(model_info, f, default_flow_style=False)

print(f"Model info saved: {info_path}")

# Inference function
def predict_road_damage(image_path, confidence_threshold=0.5):
    """Predict road damage on image"""
    results = best_model.predict(image_path, conf=confidence_threshold)
    
    predictions = []
    if len(results) > 0 and results[0].boxes is not None:
        boxes = results[0].boxes
        for box in boxes:
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
            conf = box.conf[0].cpu().numpy()
            cls = int(box.cls[0].cpu().numpy())
            
            predictions.append({
                'class': list(class_names.values())[cls],
                'confidence': float(conf),
                'bbox': [float(x1), float(y1), float(x2), float(y2)]
            })
    
    return predictions

print("Training completed!")
print(f"Best model: {best_model_path}")
print("Use predict_road_damage(image_path) for inference")